> Notebook-friendly copy of `part-I/1.7-oop-for-environmental-systems-environmental-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# Environmental Solutions

**ℹ️ Reference solutions**

Worked solutions for the real-dataset exercises in [1.7-oop-for-environmental-systems-environmental-exercises.ipynb](1.7-oop-for-environmental-systems-environmental-exercises.ipynb).

## Exercise 1: Validated earthquake events — a real dataset

The [USGS](https://earthquake.usgs.gov/) publishes a feed of every earthquake it records; this exercise uses an archived extract of one month of it, January 2023.

Combine both halves of this subchapter: a class that carries state, and exceptions that refuse to let it be built from bad data.

In [ ]:
# Pre-supplied: download and cache one month of the USGS earthquake feed (January 2023).
import pooch

quakes_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2023_01.csv",
    known_hash="sha256:14e6100a41c55772f73a34daf1cd6ea38ba0fa49a86bb62a3930868614975df2",
    fname="usgs_earthquakes_2023_01.csv",
    path=pooch.os_cache("mlees"),
)

**Step 1.** Read `quakes_path` with pandas, keeping only the `place`, `mag`, and `depth` columns. Print the resulting shape.

In [ ]:
import pandas as pd

quakes = pd.read_csv(quakes_path)[["place", "mag", "depth"]]
print(quakes.shape)

**Step 2.** Define an `EarthquakeEvent` class with instance attributes `place: str`, `magnitude: float`, and `depth_km: float`; a `__repr__` showing all three; and a method `is_shallow(self, threshold_km=70.0) -> bool` returning whether `depth_km` is below the threshold. In `__init__`, raise `ValueError` if `magnitude` or `depth_km` is negative — a real event never has either, and this is external data, not your own arithmetic, so it is an exception, not an assert.

In [ ]:
class EarthquakeEvent:
    """One recorded earthquake, validated at construction."""

    def __init__(self, place: str, magnitude: float, depth_km: float) -> None:
        if magnitude < 0.0 or depth_km < 0.0:
            raise ValueError(
                f"negative magnitude or depth: {magnitude}, {depth_km} km"
            )
        self.place = place
        self.magnitude = magnitude
        self.depth_km = depth_km

    def __repr__(self) -> str:
        return (f"EarthquakeEvent(place={self.place!r}, "
                f"magnitude={self.magnitude}, depth_km={self.depth_km})")

    def is_shallow(self, threshold_km: float = 70.0) -> bool:
        """Return whether the event is shallower than the threshold."""
        return self.depth_km < threshold_km

**Step 3.** Confirm the validation works: build one valid `EarthquakeEvent` and print it, then show that `EarthquakeEvent("test", -1.0, 10.0)` raises `ValueError`.

In [ ]:
event = EarthquakeEvent("10 km NE of Somewhere", 4.2, 35.0)
print(event)
print("shallow?", event.is_shallow())

try:
    EarthquakeEvent("test", -1.0, 10.0)
except ValueError as err:
    print("refused, as it should be:", err)

**Step 4.** Loop over the three columns from Step 1 with `zip`, building one `EarthquakeEvent` per row inside a `try/except ValueError`. Collect the successfully built events in a list, and count how many rows were rejected instead of stopping the loop.

In [ ]:
events = []
n_rejected = 0
for place, mag, depth in zip(quakes["place"], quakes["mag"], quakes["depth"]):
    try:
        events.append(EarthquakeEvent(place, mag, depth))
    except ValueError:
        n_rejected += 1

print(len(events), "built |", n_rejected, "rejected")

# nothing is rejected here: this month of the USGS feed happens to carry no negative
# magnitude or depth. The guard is still what makes the loop safe to run on a month
# that does -- step 3 is where it was actually exercised.

**Step 5.** Using `is_shallow()`, count how many events in your list are shallow. Separately, find the single deepest event with an explicit loop that tracks a running "deepest so far" variable — the same pattern `StationNetwork.coldest_station` uses in the lecture, not `max()`.

In [ ]:
n_shallow = 0
for event in events:
    if event.is_shallow():
        n_shallow += 1

deepest = None
for event in events:
    if deepest is None or event.depth_km > deepest.depth_km:
        deepest = event

print("shallow events:", n_shallow)
print("deepest:", deepest)

**Step 6.** Print a short summary: the number of events built, the number of rows rejected, the number of shallow events, and the deepest event's repr.

In [ ]:
print(len(events), "events,", n_rejected, "rejected,", n_shallow, "shallow")
print(deepest)

## Exercise 2: The solar system, continued

Exercise 2 of [1.2](1.2-data-structures-and-control-flow-environmental-exercises.ipynb) left the solar system
as a dictionary: `planet_masses_1e24kg`, mapping a name to a mass in units of 10^24 kg. A
dictionary keeps the name and the mass together only by convention — pass the mass somewhere on
its own and the name is gone, which is exactly what went wrong in that exercise's bonus question.

Now that classes are available, finish the job: give a planet a type, so that its name and its mass
travel as one object and the comparison you want to make becomes a method on it.

The dictionary is rebuilt below so this exercise stands on its own.

In [ ]:
# Pre-supplied: the dictionary from Exercise 2 of 1.2's environmental exercises, rebuilt here.
planet_masses_1e24kg = {
    "mercury": 0.330,
    "venus": 4.87,
    "earth": 5.97,
    "mars": 0.642,
    "jupiter": 1898.0,
    "saturn": 568.0,
    "uranus": 86.8,
    "neptune": 102.0,
}

**Q12) Write a class `Planet` with two attributes, `name` and `mass_1e24kg`.**

Create one instance for Earth (5.97) and one for Jupiter (1898), then print each one's name and
mass to check they were stored.

Hint: give `__init__` type annotations, as 1.7's lecture does — `name: str`, `mass_1e24kg: float`.
They are not enforced at runtime, but they are the only place the unit is written down.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        self.name = name
        self.mass_1e24kg = mass_1e24kg

earth = Planet("earth", planet_masses_1e24kg["earth"])
jupiter = Planet("jupiter", planet_masses_1e24kg["jupiter"])

print(earth.name, earth.mass_1e24kg)          # earth 5.97
print(jupiter.name, jupiter.mass_1e24kg)      # jupiter 1898.0

**Q13) Add a method `is_light` that compares a planet against Jupiter.**

It should report `True` when the planet is strictly lighter than Jupiter, `False` when it is
strictly heavier, and the string `"same mass!"` when the two are equal. Ask it whether Earth is
lighter than Jupiter, and whether Jupiter is lighter than itself.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        self.name = name
        self.mass_1e24kg = mass_1e24kg

    def is_light(self):
        # as asked: prints rather than returns, and mixes bool with str
        if self.mass_1e24kg < planet_masses_1e24kg["jupiter"]:
            print(True)
        elif self.mass_1e24kg > planet_masses_1e24kg["jupiter"]:
            print(False)
        else:
            print("same mass!")

In [ ]:
Planet("earth", planet_masses_1e24kg["earth"]).is_light()        # True
Planet("jupiter", planet_masses_1e24kg["jupiter"]).is_light()    # same mass!

In [ ]:
# the method prints its answer and returns nothing, so the caller gets None
print(Planet("earth", planet_masses_1e24kg["earth"]).is_light())

**Q14) Now fix what Q13 asked you to build.**

The method you just wrote has two problems that 1.7's own material names directly.

1. It *prints* its answer instead of returning it, so a caller cannot use the result —
   `if earth.is_light():` will not do what you expect.
2. It returns a `bool` on two branches and a `str` on the third, so the caller has to test the
   type of the answer before trusting it.

Rewrite it as `compare_to_jupiter()` returning one of three strings — `"lighter"`, `"heavier"`,
`"same"` — and have the caller do the printing. Then loop over every planet in
`planet_masses_1e24kg`, build a `Planet` for each, and print the verdict for all eight.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        self.name = name
        self.mass_1e24kg = mass_1e24kg

    def compare_to_jupiter(self) -> str:
        # one return type on every branch, and the caller decides what to do with it
        jupiter_mass_1e24kg = planet_masses_1e24kg["jupiter"]
        if self.mass_1e24kg < jupiter_mass_1e24kg:
            return "lighter"
        if self.mass_1e24kg > jupiter_mass_1e24kg:
            return "heavier"
        return "same"

In [ ]:
for name, mass_1e24kg in planet_masses_1e24kg.items():
    planet = Planet(name, mass_1e24kg)
    print(f"{planet.name:>8}: {planet.compare_to_jupiter()}")

**Q15) Give the class the invariant it is missing.**

A negative mass is not a planet. Raise a `ValueError` from `__init__` when `mass_1e24kg` is not
strictly positive, and show both that a valid planet still constructs and that
`Planet("nonsense", -1.0)` fails. Add a `__repr__` so a `Planet` prints readably.

This is the same shape as Exercise 1's `EarthquakeEvent`: validate at construction, so that every
object that exists is one you can trust.

In [ ]:
class Planet:
    def __init__(self, name: str, mass_1e24kg: float) -> None:
        if mass_1e24kg <= 0.0:
            raise ValueError(f"mass must be positive, got {mass_1e24kg}")
        self.name = name
        self.mass_1e24kg = mass_1e24kg

    def __repr__(self) -> str:
        return f"Planet(name={self.name!r}, mass_1e24kg={self.mass_1e24kg})"

    def compare_to_jupiter(self) -> str:
        jupiter_mass_1e24kg = planet_masses_1e24kg["jupiter"]
        if self.mass_1e24kg < jupiter_mass_1e24kg:
            return "lighter"
        if self.mass_1e24kg > jupiter_mass_1e24kg:
            return "heavier"
        return "same"

In [ ]:
print(Planet("earth", 5.97))        # Planet(name='earth', mass_1e24kg=5.97)

In [ ]:
try:
    Planet("nonsense", -1.0)
except ValueError as err:
    print("rejected:", err)         # rejected: mass must be positive, got -1.0

**ℹ️ What you just did**

You turned a dictionary into a type. The name and the mass are now one object rather than a key
and a value that happen to line up, the comparison is a method rather than a loose function that
has to be handed the right dictionary, and the constructor refuses to build a planet that could not
exist.

The Q13-to-Q14 step is worth a second look. Q13 asked for exactly what the exercise's earlier
edition asked for, and the result is a method that prints instead of returning and changes its
return type between branches. Both are easy to write and hard to use, and neither shows up until
something calls the method — which is why 1.7 spends its second half on making failures visible
early rather than on making code shorter.